# MS-Swift Qwen3.5-27B Colab Training + Benchmark Bundle

This notebook is for future FineTree training runs on Colab, with no dependency on the FineTree repo.

Workflow:

1. install dependencies
2. configure the run and save metadata before training
3. fine-tune with `swift sft`
4. keep epoch checkpoints during training without trainer-side validation loss
5. push the final adapter checkpoint to Hugging Face
6. run `transformers` inference once for the final checkpoint or another configured target
7. emit a self-contained benchmark bundle: `info.json`, `logging.jsonl`, `predictions/`, and a zip archive

Supported inference targets:

- local adapter
- remote adapter
- local full model
- remote full model

Use the separate `qwen35_finetree_v21_infer_only.ipynb` notebook for one-time backfills of already-trained merged models such as `asafd60/Qwen3.5-27B-FineTree-V2.1`.


In [ ]:
# Exact dependency flow from the source notebook, minus flash-attention.
!pip install --upgrade pip
!python -m pip install -U uv
!uv pip install --system -U transformers trl peft datasets accelerate bitsandbytes huggingface_hub pillow qwen_vl_utils
!uv pip install ms-swift==v4.0.0


In [ ]:
import json
import math
import os
import platform
import re
import shlex
import shutil
import socket
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
from zoneinfo import ZoneInfo

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def israel_now_iso():
    return datetime.now(ZoneInfo("Asia/Jerusalem")).isoformat()

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def run_command(label, command, env_overrides=None):
    env = dict(os.environ)
    env.update(env_overrides or {})
    rendered = " ".join(shlex.quote(str(part)) for part in command)
    print(f"\n[{label}] {rendered}")
    process = subprocess.Popen(
        [str(part) for part in command],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-200:]).strip()
        raise RuntimeError(f"{label} failed with exit code {return_code}. Last output:\n{tail}")

def cli_value(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def append_cli_args(command, arg_map):
    out = list(command)
    for key, value in arg_map.items():
        if value is None:
            continue
        cli_key = "val_dataset" if key == "validation_dataset" else key
        out.extend([f"--{cli_key}", cli_value(value)])
    return out

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return path

def load_logging_rows(path):
    rows = []
    for line_number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        payload = json.loads(line)
        if not isinstance(payload, dict):
            raise ValueError(f"logging.jsonl line {line_number} must be a JSON object")
        rows.append(payload)
    if not rows:
        raise ValueError("logging.jsonl did not contain any rows")
    return rows

def collect_environment_snapshot(active_env):
    gpu_names = []
    gpu_count = 0
    torch_version = None
    cuda_runtime_version = None
    try:
        import torch
        torch_version = getattr(torch, "__version__", None)
        cuda_runtime_version = getattr(getattr(torch, "version", None), "cuda", None)
        if torch.cuda.is_available():
            gpu_count = int(torch.cuda.device_count())
            gpu_names = [str(torch.cuda.get_device_name(i)) for i in range(gpu_count)]
    except Exception:
        pass
    nvidia_smi = None
    try:
        completed = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            check=False,
            capture_output=True,
            text=True,
        )
        if completed.returncode == 0:
            nvidia_smi = completed.stdout.strip() or None
            if not gpu_names and nvidia_smi:
                gpu_names = [line.split(",", 1)[0].strip() for line in nvidia_smi.splitlines() if line.strip()]
                gpu_count = len(gpu_names)
    except Exception:
        pass
    return {
        "platform": platform.platform(),
        "platform_machine": platform.machine(),
        "python_version": platform.python_version(),
        "python_executable": sys.executable,
        "hostname": socket.gethostname(),
        "torch_version": torch_version,
        "cuda_runtime_version": cuda_runtime_version,
        "ms_swift_version": package_version("ms-swift") or package_version("swift"),
        "transformers_version": package_version("transformers"),
        "cuda_visible_devices": active_env.get("CUDA_VISIBLE_DEVICES"),
        "max_pixels_env": active_env.get("MAX_PIXELS"),
        "gpu_count": gpu_count,
        "gpu_names": gpu_names,
        "gpu_used": " | ".join(gpu_names) if gpu_names else None,
        "nvidia_smi": nvidia_smi,
    }

def parse_step_progress(row):
    text = str(row.get("global_step/max_steps") or "").strip()
    if "/" not in text:
        return None, None
    left, right = text.split("/", 1)
    try:
        return int(left), int(right)
    except ValueError:
        return None, None

def select_best_checkpoint(output_dir, logging_rows):
    output_dir = Path(output_dir)
    checkpoint_step_map = {}
    for path in output_dir.iterdir():
        if path.is_dir():
            match = re.fullmatch(r"checkpoint-(\d+)", path.name)
            if match:
                checkpoint_step_map[int(match.group(1))] = path
    candidates = []
    for row in logging_rows:
        if "eval_loss" not in row:
            continue
        try:
            eval_loss = float(row["eval_loss"])
        except Exception:
            continue
        global_step, max_steps = parse_step_progress(row)
        if global_step is None or global_step not in checkpoint_step_map:
            continue
        epoch = row.get("epoch")
        epoch_value = float(epoch) if isinstance(epoch, (int, float)) else None
        candidates.append({
            "checkpoint_name": checkpoint_step_map[global_step].name,
            "checkpoint_path": str(checkpoint_step_map[global_step]),
            "global_step": global_step,
            "max_steps": max_steps,
            "epoch": epoch_value,
            "eval_loss": eval_loss,
            "eval_token_acc": float(row["eval_token_acc"]) if isinstance(row.get("eval_token_acc"), (int, float)) else None,
            "selection_metric": "eval_loss",
            "selection_reason": "lowest_eval_loss",
            "adapter_only": True,
        })
    if candidates:
        return sorted(
            candidates,
            key=lambda item: (float(item["eval_loss"]), -(item["epoch"] if item["epoch"] is not None else -1.0), -int(item["global_step"])),
        )[0]
    if checkpoint_step_map:
        final_step = max(checkpoint_step_map)
        return {
            "checkpoint_name": checkpoint_step_map[final_step].name,
            "checkpoint_path": str(checkpoint_step_map[final_step]),
            "global_step": final_step,
            "max_steps": None,
            "epoch": None,
            "eval_loss": None,
            "eval_token_acc": None,
            "selection_metric": "fallback",
            "selection_reason": "final_checkpoint",
            "adapter_only": True,
        }
    raise FileNotFoundError(f"No checkpoint-* directories found under {output_dir}")

def export_adapter_to_hub(adapter_path, hub_model_id, export_dir, swift_env, base_model, hf_token):
    if not hf_token:
        print(f"Skipping adapter push for {hub_model_id}: HF_TOKEN is not set")
        return False
    export_dir = Path(export_dir)
    if export_dir.exists():
        shutil.rmtree(export_dir)
    command = [
        "swift",
        "export",
        "--model",
        base_model,
        "--adapters",
        str(adapter_path),
        "--push_to_hub",
        "true",
        "--use_hf",
        "true",
        "--hub_model_id",
        hub_model_id,
        "--hub_token",
        hf_token,
        "--output_dir",
        str(export_dir),
    ]
    run_command(f"swift export {hub_model_id}", command, env_overrides=swift_env)
    return True

def maybe_parse_json_string(text):
    stripped = str(text).strip()
    if not stripped.startswith("{"):
        return None
    try:
        payload = json.loads(stripped)
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None

def maybe_extract_json_object(text):
    if not isinstance(text, str):
        return None
    parsed = maybe_parse_json_string(text)
    if parsed is not None:
        return parsed
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        payload = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None

def extract_prediction_payload(row):
    if isinstance(row, dict):
        if isinstance(row.get("pages"), list) or isinstance(row.get("facts"), list):
            return row
        for key in ("prediction_json", "parsed_prediction", "prediction", "response", "generated_text", "output", "text"):
            value = row.get(key)
            if isinstance(value, dict) and (isinstance(value.get("pages"), list) or isinstance(value.get("facts"), list)):
                return value
            parsed = maybe_extract_json_object(value)
            if parsed is not None:
                return parsed
        messages = row.get("messages")
        if isinstance(messages, list):
            for message in reversed(messages):
                if not isinstance(message, dict):
                    continue
                parsed = maybe_extract_json_object(message.get("content"))
                if parsed is not None:
                    return parsed
    if isinstance(row, str):
        parsed = maybe_extract_json_object(row)
        if parsed is not None:
            return parsed
    raise ValueError("Could not extract a benchmark JSON payload from inference output. Refusing to fall back to labels or other non-prediction fields.")

def materialize_prediction_json_files(result_jsonl_path, output_dir, prefix="pred_"):
    result_jsonl_path = Path(result_jsonl_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written_paths = []
    for index, line in enumerate(result_jsonl_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        payload = json.loads(line)
        prediction = extract_prediction_payload(payload)
        out_path = output_dir / f"{prefix}{index:04d}.json"
        out_path.write_text(json.dumps(prediction, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        written_paths.append(out_path)
    if not written_paths:
        raise ValueError(f"No prediction files were written from {result_jsonl_path}")
    return written_paths

def sync_prediction_json_files(result_jsonl_path, output_dir, processed_line_count=0, prefix="pred_"):
    result_jsonl_path = Path(result_jsonl_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if not result_jsonl_path.exists():
        return processed_line_count, []
    lines = result_jsonl_path.read_text(encoding="utf-8").splitlines()
    new_paths = []
    for line_index, line in enumerate(lines[processed_line_count:], start=processed_line_count + 1):
        if not line.strip():
            continue
        payload = json.loads(line)
        prediction = extract_prediction_payload(payload)
        out_path = output_dir / f"{prefix}{line_index:04d}.json"
        out_path.write_text(json.dumps(prediction, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        new_paths.append(out_path)
        print(f"[prediction] wrote {out_path.name}")
    return len(lines), new_paths

def run_command_stream_predictions(label, command, result_jsonl_path, output_dir, env_overrides=None, poll_interval_seconds=2.0):
    env = dict(os.environ)
    env.update(env_overrides or {})
    rendered = " ".join(shlex.quote(str(part)) for part in command)
    print(f"\n[{label}] {rendered}")
    process = subprocess.Popen(
        [str(part) for part in command],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    output_lines = []

    def forward_stdout():
        for line in process.stdout:
            print(line, end="")
            output_lines.append(line)

    stdout_thread = threading.Thread(target=forward_stdout, daemon=True)
    stdout_thread.start()

    processed_line_count = 0
    written_paths = []
    while stdout_thread.is_alive() or process.poll() is None:
        processed_line_count, new_paths = sync_prediction_json_files(
            result_jsonl_path,
            output_dir,
            processed_line_count=processed_line_count,
        )
        written_paths.extend(new_paths)
        time.sleep(poll_interval_seconds)

    stdout_thread.join()
    processed_line_count, new_paths = sync_prediction_json_files(
        result_jsonl_path,
        output_dir,
        processed_line_count=processed_line_count,
    )
    written_paths.extend(new_paths)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-200:]).strip()
        raise RuntimeError(f"{label} failed with exit code {return_code}. Last output:\n{tail}")
    if not written_paths:
        raise ValueError(f"No prediction files were written from {result_jsonl_path}")
    return written_paths

BENCHMARK_METADATA_FIELDS = [
    "checkpoint_name", "model", "tuner_type", "dataset", "validation_dataset", "use_hf", "freeze_vit",
    "vit_lr", "freeze_aligner", "enable_thinking", "add_non_thinking_prefix", "torch_dtype",
    "num_train_epochs", "per_device_train_batch_size", "per_device_eval_batch_size", "learning_rate",
    "lora_rank", "lora_alpha", "target_modules", "gradient_accumulation_steps", "output_dir",
    "eval_steps", "save_steps", "save_total_limit", "logging_steps", "max_length", "warmup_ratio",
    "dataset_num_proc", "dataloader_num_workers", "eval_on_start", "max_pixels", "temperature",
    "gradient_checkpointing", "gradient_checkpointing_kwargs", "truncation_strategy",
    "CUDA_VISIBLE_DEVICES", "MAX_PIXELS", "gpu_used", "torch_env_used", "platform"
]

def build_benchmark_model_metadata(training_args, environment, checkpoint_name, metadata_overrides=None):
    metadata_out = {key: None for key in BENCHMARK_METADATA_FIELDS}
    metadata_out.update({key: training_args.get(key) for key in BENCHMARK_METADATA_FIELDS if key in training_args})
    metadata_out["checkpoint_name"] = checkpoint_name
    metadata_out["validation_dataset"] = training_args.get("validation_dataset") or training_args.get("val_dataset")
    metadata_out["CUDA_VISIBLE_DEVICES"] = training_args.get("CUDA_VISIBLE_DEVICES") or environment.get("cuda_visible_devices")
    metadata_out["MAX_PIXELS"] = training_args.get("MAX_PIXELS") or environment.get("max_pixels_env")
    metadata_out["gpu_used"] = environment.get("gpu_used")
    metadata_out["torch_env_used"] = " | ".join(
        value for value in [
            f"torch {environment.get('torch_version')}" if environment.get("torch_version") else None,
            f"cuda {environment.get('cuda_runtime_version')}" if environment.get("cuda_runtime_version") else None,
            f"python {environment.get('python_version')}" if environment.get("python_version") else None,
        ]
        if value
    ) or None
    metadata_out["platform"] = environment.get("platform")
    if metadata_overrides:
        metadata_out.update(dict(metadata_overrides))
    return metadata_out

def build_submission_info_payload(model_metadata, training_args, environment, run, selected_checkpoint, artifacts):
    return {
        "model_metadata": dict(model_metadata),
        "training_args": dict(training_args),
        "environment": dict(environment),
        "run": dict(run),
        "selected_checkpoint": dict(selected_checkpoint),
        "artifacts": dict(artifacts),
    }

def _reference_label(value):
    text = str(value).rstrip("/")
    if not text:
        return "model"
    return text.split("/")[-1]

def resolve_inference_target(target_config, selected_checkpoint, base_model, default_remote_adapter_repo=None):
    config = dict(target_config or {})
    mode = str(config.get("mode") or "local_adapter").strip()
    if mode == "local_adapter":
        adapter_ref = config.get("adapter") or selected_checkpoint["checkpoint_path"]
        model_ref = config.get("base_model") or base_model
        use_hf = bool(config.get("use_hf", True))
        label = str(config.get("label") or Path(str(adapter_ref)).name)
    elif mode == "remote_adapter":
        adapter_ref = config.get("adapter") or default_remote_adapter_repo
        if not adapter_ref:
            raise ValueError("remote_adapter mode requires an adapter repo id or a pushed best adapter checkpoint")
        model_ref = config.get("base_model") or base_model
        use_hf = True if config.get("use_hf") is None else bool(config.get("use_hf"))
        label = str(config.get("label") or _reference_label(adapter_ref))
    elif mode == "local_full_model":
        model_ref = config.get("model")
        if not model_ref:
            raise ValueError("local_full_model mode requires a local model path")
        adapter_ref = None
        use_hf = False if config.get("use_hf") is None else bool(config.get("use_hf"))
        label = str(config.get("label") or Path(str(model_ref)).name)
    elif mode == "remote_full_model":
        model_ref = config.get("model")
        if not model_ref:
            raise ValueError("remote_full_model mode requires a remote model repo id")
        adapter_ref = None
        use_hf = True if config.get("use_hf") is None else bool(config.get("use_hf"))
        label = str(config.get("label") or _reference_label(model_ref))
    else:
        raise ValueError("Unsupported inference target mode: " + mode)
    return {
        "mode": mode,
        "label": label,
        "use_hf": use_hf,
        "model_ref": str(model_ref),
        "adapter_ref": None if adapter_ref is None else str(adapter_ref),
        "base_model": str(base_model),
    }

def resolve_torch_dtype(dtype_name):
    import torch
    mapping = {
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float16": torch.float16,
        "fp16": torch.float16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }
    return mapping.get(str(dtype_name or "bfloat16").lower(), torch.bfloat16)

def load_inference_dataset(dataset_id):
    from datasets import load_dataset
    dataset = load_dataset(dataset_id)
    for split_name in ("train", "validation", "test"):
        if split_name in dataset:
            return dataset[split_name]
    raise RuntimeError(f"Dataset {dataset_id} does not expose a usable split")

def extract_sample_images(sample):
    images = sample.get("images")
    if images is None:
        image = sample.get("image")
        images = [] if image is None else [image]
    elif not isinstance(images, list):
        images = [images]
    if not images:
        raise ValueError("Inference sample is missing image content")
    return images

def build_inference_messages(sample, images):
    messages = sample.get("messages")
    if isinstance(messages, list) and messages:
        rebuilt = []
        image_index = 0
        for message in messages:
            if not isinstance(message, dict):
                continue
            role = str(message.get("role") or "user")
            if role == "assistant":
                continue
            content = message.get("content")
            if isinstance(content, str):
                rebuilt.append({"role": role, "content": [{"type": "text", "text": content}]})
                continue
            if not isinstance(content, list):
                continue
            rebuilt_content = []
            for item in content:
                if not isinstance(item, dict):
                    continue
                item_type = item.get("type")
                if item_type == "image":
                    if image_index >= len(images):
                        raise ValueError("Message image placeholders exceed available images")
                    rebuilt_content.append({"type": "image", "image": images[image_index]})
                    image_index += 1
                elif item_type == "text":
                    rebuilt_content.append({"type": "text", "text": item.get("text", "")})
            if rebuilt_content:
                rebuilt.append({"role": role, "content": rebuilt_content})
        if rebuilt:
            if image_index < len(images):
                for message in rebuilt:
                    if message.get("role") == "user":
                        for image in images[image_index:]:
                            message["content"].insert(0, {"type": "image", "image": image})
                        image_index = len(images)
                        break
            return rebuilt
    instruction = sample.get("instruction") or sample.get("prompt") or ""
    system_message = sample.get("system")
    fallback = []
    if system_message:
        fallback.append({"role": "system", "content": [{"type": "text", "text": system_message}]})
    fallback.append({
        "role": "user",
        "content": [{"type": "image", "image": images[0]}, {"type": "text", "text": str(instruction)}],
    })
    return fallback

def run_transformers_dataset_inference(target, dataset_id, raw_predictions_path, output_dir, inference_config):
    import torch
    from peft import PeftModel
    from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for transformers inference")

    dtype = resolve_torch_dtype(inference_config.get("torch_dtype"))
    processor_ref = target["base_model"] if target.get("adapter_ref") else target["model_ref"]
    processor_kwargs = {"trust_remote_code": True}
    max_pixels = inference_config.get("max_pixels")
    if max_pixels not in (None, ""):
        processor_kwargs["min_pixels"] = int(max_pixels)
        processor_kwargs["max_pixels"] = int(max_pixels)
    processor = AutoProcessor.from_pretrained(processor_ref, **processor_kwargs)
    model = Qwen3_5ForConditionalGeneration.from_pretrained(
        target["base_model"] if target.get("adapter_ref") else target["model_ref"],
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    if target.get("adapter_ref"):
        model = PeftModel.from_pretrained(model, target["adapter_ref"])
    model.eval()

    eval_split = load_inference_dataset(dataset_id)
    raw_predictions_path = Path(raw_predictions_path)
    output_dir = Path(output_dir)
    raw_predictions_path.parent.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    for existing in output_dir.glob("pred_*.json"):
        existing.unlink()
    if raw_predictions_path.exists():
        raw_predictions_path.unlink()

    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
    batch_size = int(inference_config.get("per_device_eval_batch_size") or 1)
    max_new_tokens = int(inference_config.get("max_new_tokens") or 2048)
    enable_thinking = bool(inference_config.get("enable_thinking", False))
    written_paths = []

    with raw_predictions_path.open("w", encoding="utf-8") as raw_handle:
        prediction_index = 0
        for batch_start in range(0, len(eval_split), batch_size):
            batch_stop = min(batch_start + batch_size, len(eval_split))
            batch_samples = [eval_split[index] for index in range(batch_start, batch_stop)]
            chat_texts = []
            batch_images = []
            batch_metadata = []
            for sample in batch_samples:
                sample_images = extract_sample_images(sample)
                if len(sample_images) != 1:
                    raise ValueError("Only single-image samples are supported in this notebook")
                messages = build_inference_messages(sample, sample_images)
                chat_text = processor.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=enable_thinking,
                )
                chat_texts.append(chat_text)
                batch_images.append(sample_images[0])
                batch_metadata.append({
                    "document_id": sample.get("document_id"),
                    "annotation_file": sample.get("annotation_file"),
                })

            inputs = processor(text=chat_texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
            prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
            for row_index, prompt_length in enumerate(prompt_lengths):
                prediction_index += 1
                output_tokens = generated[row_index, int(prompt_length):]
                response_text = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()
                prediction = extract_prediction_payload({"response": response_text})
                prediction_path = output_dir / f"pred_{prediction_index:04d}.json"
                prediction_path.write_text(json.dumps(prediction, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
                raw_payload = {
                    "sample_index": prediction_index,
                    "response": response_text,
                    "prediction_file": prediction_path.name,
                    **batch_metadata[row_index],
                }
                raw_handle.write(json.dumps(raw_payload, ensure_ascii=False) + "\n")
                raw_handle.flush()
                written_paths.append(prediction_path)
                print(f"[prediction] wrote {prediction_path.name}")
            print(f"[batch] completed {batch_stop}/{len(eval_split)}")
    if not written_paths:
        raise ValueError(f"No prediction files were written for dataset {dataset_id}")
    return written_paths


In [ ]:
BASE_MODEL = "Qwen/Qwen3.5-27B"
DATASET_ID = "asafd60/FineTree_V2-approved-no-bbox-train"
VAL_DATASET_ID = "asafd60/FineTree_V2-approved-no-bbox-validation"
HF_TOKEN = os.getenv("HF_TOKEN")
HF_ADAPTER_REPO_PREFIX = "asafd60/Qwen3.5-27B-finetree-lora"

SWIFT_ENV = {
    "CUDA_VISIBLE_DEVICES": "0",
    "MAX_PIXELS": "1000000",
}

INFERENCE_BATCH_SIZE = 1

RUN_ID = f"ms-swift-qwen35-{time.strftime('%Y%m%d-%H%M%S')}"
COLAB_ROOT = Path("/content")
RUN_OUTPUT_DIR = COLAB_ROOT / "output" / "Qwen3.5-27B" / RUN_ID
BENCHMARK_BUNDLE_DIR = RUN_OUTPUT_DIR / "benchmark_submission"
PREDICTIONS_DIR = BENCHMARK_BUNDLE_DIR / "predictions"
RAW_PREDICTIONS_PATH = BENCHMARK_BUNDLE_DIR / "raw_predictions.jsonl"
RUN_ARCHIVE_BASE = RUN_OUTPUT_DIR / f"{RUN_ID}_benchmark_submission"

# Keep save_total_limit unset so the final checkpoint is always available for bundling.
TRAINING_CONFIG = {
    "model": BASE_MODEL,
    "tuner_type": "lora",
    "use_hf": True,
    "dataset": DATASET_ID,
    "load_from_cache_file": True,
    "freeze_vit": False,
    "vit_lr": 5e-6,
    "freeze_aligner": False,
    "enable_thinking": False,
    "add_non_thinking_prefix": True,
    "torch_dtype": "bfloat16",
    "num_train_epochs": 8,
    "per_device_train_batch_size": 1,
    "learning_rate": 1e-4,
    "lora_rank": 4,
    "lora_alpha": 16,
    "target_modules": "all-linear",
    "gradient_accumulation_steps": 16,
    "output_dir": str(RUN_OUTPUT_DIR),
    "save_steps": None,
    "save_total_limit": None,
    "logging_steps": 5,
    "max_length": 6250,
    "warmup_ratio": 0.03,
    "dataset_num_proc": 3,
    "dataloader_num_workers": 3,
    "max_pixels": 1_000_000,
    "temperature": 0,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "truncation_strategy": "right",
    "save_strategy": "epoch",
    "push_to_hub": False,
    "CUDA_VISIBLE_DEVICES": SWIFT_ENV["CUDA_VISIBLE_DEVICES"],
    "MAX_PIXELS": int(SWIFT_ENV["MAX_PIXELS"]),
}

INFER_CONFIG = {
    "enable_thinking": TRAINING_CONFIG["enable_thinking"],
    "add_non_thinking_prefix": TRAINING_CONFIG["add_non_thinking_prefix"],
    "temperature": TRAINING_CONFIG["temperature"],
    "max_new_tokens": 12_000,
    "per_device_eval_batch_size": INFERENCE_BATCH_SIZE,
    "torch_dtype": TRAINING_CONFIG["torch_dtype"],
    "max_pixels": TRAINING_CONFIG["max_pixels"],
}

# Supported modes:
# - local_adapter: benchmark the final local adapter checkpoint from this run
# - remote_adapter: benchmark an adapter repo on Hugging Face
# - local_full_model: benchmark a local full-model path
# - remote_full_model: benchmark a remote full-model repo on Hugging Face
#
# Defaults to the final local adapter checkpoint. To switch targets, edit this
# cell after training and before the inference/bundle cell.
INFERENCE_TARGET_CONFIG = {
    "mode": "local_adapter",
    "base_model": BASE_MODEL,
    "adapter": None,
    "model": None,
    "use_hf": True,
    "label": None,
}

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

environment = collect_environment_snapshot({**os.environ, **SWIFT_ENV})
run_info = {
    "run_id": RUN_ID,
    "base_model": BASE_MODEL,
    "dataset": DATASET_ID,
    "validation_dataset": VAL_DATASET_ID,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "benchmark_bundle_dir": str(BENCHMARK_BUNDLE_DIR),
    "training_started_at_utc": utc_now_iso(),
    "training_started_at_israel": israel_now_iso(),
}

write_json(RUN_OUTPUT_DIR / "run_info.json", {
    "training_args": TRAINING_CONFIG,
    "environment": environment,
    "run": run_info,
})

print(f"Run output dir: {RUN_OUTPUT_DIR}")
print(f"Bundle dir: {BENCHMARK_BUNDLE_DIR}")
print(json.dumps({"run": run_info, "environment": environment}, ensure_ascii=False, indent=2))


In [ ]:
train_started = time.time()
sft_command = append_cli_args(["swift", "sft"], TRAINING_CONFIG)
run_command("swift sft", sft_command, env_overrides=SWIFT_ENV)
train_finished = time.time()

run_info.update({
    "training_finished_at_utc": utc_now_iso(),
    "training_finished_at_israel": israel_now_iso(),
    "training_duration_seconds": round(train_finished - train_started, 2),
})
write_json(RUN_OUTPUT_DIR / "run_info.json", {
    "training_args": TRAINING_CONFIG,
    "environment": environment,
    "run": run_info,
})


In [ ]:
logging_path = RUN_OUTPUT_DIR / "logging.jsonl"
logging_rows = load_logging_rows(logging_path)

checkpoint_paths = sorted(
    [path for path in RUN_OUTPUT_DIR.glob("checkpoint-*") if path.is_dir()],
    key=lambda path: int(path.name.split("-")[-1]),
)
if not checkpoint_paths:
    raise FileNotFoundError(f"No checkpoint-* directories found under {RUN_OUTPUT_DIR}")
final_checkpoint_path = checkpoint_paths[-1]
selected_checkpoint = {
    "checkpoint_name": final_checkpoint_path.name,
    "checkpoint_path": str(final_checkpoint_path),
    "selection_metric": "final_checkpoint",
    "selection_reason": "generation_only_no_trainer_eval",
    "adapter_only": True,
}

selected_checkpoint["pushed_adapter_to_hub"] = export_adapter_to_hub(
    selected_checkpoint["checkpoint_path"],
    f"{HF_ADAPTER_REPO_PREFIX}-final",
    RUN_OUTPUT_DIR / "hub_export_final",
    SWIFT_ENV,
    BASE_MODEL,
    HF_TOKEN,
)
selected_checkpoint["hub_model_id"] = f"{HF_ADAPTER_REPO_PREFIX}-final" if selected_checkpoint["pushed_adapter_to_hub"] else None
selected_checkpoint["final_checkpoint_name"] = final_checkpoint_path.name

print(json.dumps(selected_checkpoint, ensure_ascii=False, indent=2))


In [ ]:
resolved_inference_target = resolve_inference_target(
    INFERENCE_TARGET_CONFIG,
    selected_checkpoint=selected_checkpoint,
    base_model=BASE_MODEL,
    default_remote_adapter_repo=selected_checkpoint.get("hub_model_id"),
)

selected_checkpoint["evaluation_target_mode"] = resolved_inference_target["mode"]
selected_checkpoint["evaluation_target_label"] = resolved_inference_target["label"]
selected_checkpoint["evaluation_target_model_ref"] = resolved_inference_target["model_ref"]
selected_checkpoint["evaluation_target_adapter_ref"] = resolved_inference_target["adapter_ref"]
selected_checkpoint["evaluation_target_use_hf"] = resolved_inference_target["use_hf"]

infer_config = dict(INFER_CONFIG)

run_info.update({
    "inference_started_at_utc": utc_now_iso(),
    "inference_started_at_israel": israel_now_iso(),
})

infer_started = time.time()
written_predictions = run_transformers_dataset_inference(
    resolved_inference_target,
    VAL_DATASET_ID,
    RAW_PREDICTIONS_PATH,
    PREDICTIONS_DIR,
    infer_config,
)
infer_finished = time.time()

run_info.update({
    "inference_finished_at_utc": utc_now_iso(),
    "inference_finished_at_israel": israel_now_iso(),
    "inference_duration_seconds": round(infer_finished - infer_started, 2),
})
model_metadata = build_benchmark_model_metadata(
    TRAINING_CONFIG,
    environment,
    resolved_inference_target["label"],
    metadata_overrides={
        "model": resolved_inference_target["model_ref"],
        "use_hf": resolved_inference_target["use_hf"],
        "dataset": DATASET_ID,
        "validation_dataset": VAL_DATASET_ID,
        "per_device_eval_batch_size": INFERENCE_BATCH_SIZE,
    },
)

artifacts = {
    "logging_jsonl": "logging.jsonl",
    "prediction_dir": "predictions",
    "raw_predictions_jsonl": "raw_predictions.jsonl",
    "submission_archive": f"{RUN_ARCHIVE_BASE.name}.zip",
    "evaluation_target_mode": resolved_inference_target["mode"],
}

info_payload = build_submission_info_payload(
    model_metadata=model_metadata,
    training_args=TRAINING_CONFIG,
    environment=environment,
    run=run_info,
    selected_checkpoint=selected_checkpoint,
    artifacts=artifacts,
)

write_json(RUN_OUTPUT_DIR / "run_info.json", {
    "training_args": TRAINING_CONFIG,
    "environment": environment,
    "run": run_info,
    "selected_checkpoint": selected_checkpoint,
})
write_json(BENCHMARK_BUNDLE_DIR / "info.json", info_payload)
shutil.copy2(logging_path, BENCHMARK_BUNDLE_DIR / "logging.jsonl")

instructions = """
This folder is a benchmark submission bundle generated on Colab.

On the benchmark machine:
1. Download and unzip the archive.
2. Point benchmark.input_dir at this folder, or copy its contents into the benchmark input folder.
3. Make sure the benchmark YAML mappings reference prediction files such as predictions/pred_0001.json.
4. Open the benchmark UI. If info.json is present, the UI should ingest it without manual form entry.
""".strip()
(BENCHMARK_BUNDLE_DIR / "README.txt").write_text(instructions + "\n", encoding="utf-8")

archive_path = Path(shutil.make_archive(str(RUN_ARCHIVE_BASE), "zip", root_dir=RUN_OUTPUT_DIR, base_dir="benchmark_submission"))

summary = {
    "run_id": RUN_ID,
    "selected_checkpoint": selected_checkpoint["checkpoint_name"],
    "evaluation_target": resolved_inference_target,
    "prediction_files": [path.name for path in written_predictions],
    "bundle_dir": str(BENCHMARK_BUNDLE_DIR),
    "archive_path": str(archive_path),
    "archive_name": archive_path.name,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


## What The User Downloads

Download the generated zip archive from Colab. It contains:

- `info.json`
- `logging.jsonl`
- `predictions/pred_0001.json`, `predictions/pred_0002.json`, ...
- `README.txt`

That bundle can then be moved to the separate benchmark/UI machine.
